# 05 Feature Engineering

This notebook adds derived features after missing values have been imputed. The output files are still working datasets, so they stay in `../interim`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
DATASET_DIR = Path("..").resolve()
INTERIM_DIR = DATASET_DIR / "interim"

TRAIN_IMPUTED_PATH = INTERIM_DIR / "training_imputed.csv"
TEST_IMPUTED_PATH = INTERIM_DIR / "testing_imputed.csv"

In [3]:
train_df = pd.read_csv(TRAIN_IMPUTED_PATH)
test_df = pd.read_csv(TEST_IMPUTED_PATH)

print(train_df.shape, test_df.shape)

(150000, 13) (101503, 13)


In [4]:
def add_credit_features(df):
    df = df.copy()

    df["TotalPastDueEvents"] = (
        df["NumberOfTime30-59DaysPastDueNotWorse"]
        + df["NumberOfTime60-89DaysPastDueNotWorse"]
        + df["NumberOfTimes90DaysLate"]
    )

    df["HasDependents"] = (df["NumberOfDependents"] > 0).astype(int)
    df["HasRealEstateLoan"] = (df["NumberRealEstateLoansOrLines"] > 0).astype(int)
    df["HighRevolvingUtilization"] = (df["RevolvingUtilizationOfUnsecuredLines"] > 1).astype(int)
    df["DebtRatioAboveOne"] = (df["DebtRatio"] > 1).astype(int)

    df["IncomePerDependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)
    df["ApproxMonthlyDebt"] = df["DebtRatio"] * df["MonthlyIncome"]
    df["LogMonthlyIncome"] = np.log1p(df["MonthlyIncome"])

    return df

train_features = add_credit_features(train_df)
test_features = add_credit_features(test_df)

In [5]:
preview_cols = [
    "MonthlyIncome",
    "NumberOfDependents",
    "TotalPastDueEvents",
    "HasDependents",
    "HasRealEstateLoan",
    "HighRevolvingUtilization",
    "IncomePerDependent",
    "ApproxMonthlyDebt",
    "LogMonthlyIncome",
]

train_features[preview_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
MonthlyIncome,150000.0,5378.463546,13143.916703,0.0,1700.000000,4392.000000,7400.000000,3.008750e+06
NumberOfDependents,150000.0,0.737580,1.107028,0.0,0.000000,0.000000,1.000000,2.000000e+01
TotalPastDueEvents,150000.0,0.927393,12.466204,0.0,0.000000,0.000000,0.000000,2.940000e+02
HasDependents,150000.0,0.394620,0.488771,0.0,0.000000,0.000000,1.000000,1.000000e+00
HasRealEstateLoan,150000.0,0.625413,0.484018,0.0,0.000000,1.000000,1.000000,1.000000e+00
HighRevolvingUtilization,150000.0,0.022140,0.147139,0.0,0.000000,0.000000,0.000000,1.000000e+00
IncomePerDependent,150000.0,3644.430260,8148.988064,0.0,948.916667,2625.000000,5000.000000,1.794060e+06
ApproxMonthlyDebt,150000.0,1700.379553,3851.771902,0.0,26.994374,1101.806098,2510.215561,4.784506e+05
LogMonthlyIncome,150000.0,6.847610,3.458025,0.0,7.438972,8.387768,8.909370,1.491704e+01


In [6]:
train_features.to_csv(INTERIM_DIR / "training_features.csv", index=False)
test_features.to_csv(INTERIM_DIR / "testing_features.csv", index=False)

print("Saved feature datasets to interim.")

Saved feature datasets to interim.
